# Notebook 04: one-parameter gradient descent

This notebook shows the smallest possible training loop: one input, one target answer, and one trainable weight.

For the current weight, the forward calculation makes a prediction and then measures how wrong it is:

- `prediction = w * x`
- `error = prediction - target`
- `loss = error ** 2`

The loss is the squared distance from the right answer. We keep `error` as its own value because its sign tells us whether the prediction is too low or too high.

In [1]:
x = 2.0
target = 6.0
w = 1.0

prediction = w * x
error = prediction - target
loss = error ** 2

print(f"Prediction: {prediction:.2f}")
print(f"Error: {error:.2f}")
print(f"Loss: {loss:.2f}")

Prediction: 2.00
Error: -4.00
Loss: 16.00


## Derive the gradient by hand

Before changing the weight, ask how the loss depends on `w`. A **derivative** is a local change rule: it tells us how one value responds when another value moves a tiny amount.

Use the full path `w -> prediction -> error -> loss`. The extra `error` step may look obvious, but keeping it visible makes the chain rule easier to trust.

The local derivatives are:

- `d_loss/d_error = 2 * error`
- `d_error/d_prediction = 1`
- `d_prediction/d_w = x`

The **backward pass** starts from the loss and walks backward through those links. The chain rule multiplies the local derivatives:

`d_loss/d_w = d_loss/d_error * d_error/d_prediction * d_prediction/d_w`

For the starting numbers, the gradient is negative. That means a small increase in `w` should reduce the loss from this exact starting point.

In [6]:
x = 2.0
target = 6.0
w = 1.0

prediction = w * x
error = prediction - target
loss = error ** 2

d_loss_d_error = 2 * error  # because the loss is squared
d_error_d_prediction = 1.0  # because adding 1 to the prediction adds 1 to the error
d_prediction_d_w = x  # because prediction = w * x, so one extra unit of w adds x to prediction
gradient = d_loss_d_error * d_error_d_prediction * d_prediction_d_w

print(f"error: {error:.2f}")
print(f"prediction: {prediction:.2f}")
print(f"loss: {loss:.2f}")
print(f"d_loss/d_error: {d_loss_d_error:.2f}")
print(f"d_error/d_prediction: {d_error_d_prediction:.2f}")
print(f"d_prediction/d_w: {d_prediction_d_w:.2f}")
print(f"d_loss/d_w: {gradient:.2f}")

error: -4.00
prediction: 2.00
loss: 16.00
d_loss/d_error: -8.00
d_error/d_prediction: 1.00
d_prediction/d_w: 2.00
d_loss/d_w: -16.00


## The gradient is local

The value `-16.00` is not a permanent truth about the model. It is the gradient at the current weight, `w = 1.0`.

The reusable rule is:

`d_loss/d_w = 2 * (w * x - target) * x`

With `x = 2.0` and `target = 6.0`, that rule becomes:

`d_loss/d_w = 8.0 * w - 24.0`

| weight `w` | prediction | error | gradient |
|---:|---:|---:|---:|
| `1.0` | `2.0` | `-4.0` | `-16.0` |
| `2.0` | `4.0` | `-2.0` | `-8.0` |
| `3.0` | `6.0` | `0.0` | `0.0` |
| `4.0` | `8.0` | `2.0` | `8.0` |

Gradient descent recalculates the gradient after each update. That is why training is a loop: the direction and size of the next step depend on the current weight.

## Training with gradient descent

**Descent** means moving downward. Here, downward means toward a smaller loss value, because loss is the number that measures how wrong the prediction is.

Gradient descent repeats one simple update rule:

`w = w - learning_rate * gradient`

The **gradient** tells the local uphill direction for the loss at the current `w`. Subtracting the gradient moves the weight in the opposite direction, which is the local downhill direction. The **learning rate** is the step size; it scales the gradient before we subtract it, so the model moves in smaller controlled steps instead of jumping by the full gradient.

Each loop pass does the same cycle:

1. **Forward calculation:** compute `prediction`, `error`, and `loss` from the current `w`.
2. **Backward calculation:** compute the gradient from the current loss path.
3. **Print the state:** show `w`, prediction, loss, and gradient so learning is visible.
4. **Update the weight:** use the update rule to prepare `w` for the next step.

The first update is the one-step example inside the loop. At `w = 1.0`, the gradient is `-16.0`; with `learning_rate = 0.1`, the update term is `-1.6`, so the next weight becomes `1.0 - (-1.6) = 2.6`. On the next pass, the prediction is `2.6 * 2.0 = 5.2`, and the loss has dropped from `16.0` to `0.64`.

The gradient must be recomputed inside the loop because `w` changes after every update. Reusing `-16.0` would describe the old weight, not the current one.

In [9]:
x = 2.0
target = 6.0
w = 1.0
learning_rate = 0.1

for step in range(5):
    # forward pass
    prediction = w * x
    error = prediction - target
    loss = error ** 2

    # backward pass
    d_loss_d_error = 2 * error
    d_error_d_prediction = 1.0
    d_prediction_d_w = x
    gradient = d_loss_d_error * d_error_d_prediction * d_prediction_d_w

    print(
           f"step {step}: "
           f"w={w:.4f}, "
           f"prediction={prediction:.4f}, "
           f"loss={loss:.6f}, "
           f"gradient={gradient:.4f}"
    )

    w = w - learning_rate * gradient

step 0: w=1.0000, prediction=2.0000, loss=16.000000, gradient=-16.0000
step 1: w=2.6000, prediction=5.2000, loss=0.640000, gradient=-3.2000
step 2: w=2.9200, prediction=5.8400, loss=0.025600, gradient=-0.6400
step 3: w=2.9840, prediction=5.9680, loss=0.001024, gradient=-0.1280
step 4: w=2.9968, prediction=5.9936, loss=0.000041, gradient=-0.0256
